# Notebook 2: Comparing Mean and Osculating Orbital Elements

We observe the time evolution of the mean orbital elements provided in a Two-Line Element (TLE) and the osculating elements at 1-hour intervals at the same epochs, which are the SGP4-propagated mean elements with secular evolution and short- as well as long-period periodic variational corrections appended. This helps better visualise what the mean orbital elements capture, compared to the osculating elements captured by the instantaneous satellite states that account for perturbations.

In this notebook, I will focus on comparing the mean and osculating orbital elements specifically for LAGEOS-1, a Medium Earth Orbit (MEO) satellite, where the TLEs were sourced from [Space-Track](www.space-track.org) for the year 2025. Hence, the annual evolution for the year 2025 will be investigated.

### Extracting 2025 TLE Elements for LAGEOS-1

I begin by first extracting the TLE elements for LAGEOS-1 for every TLE scraped between January 1st and December 31st 2025. The TLEs are first sorted by epoch, removing duplicates and retaining the TLE with the latest creation date for more precise ephemeris.

In [ ]:
# General Imports:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

# Establish Java path to remove error:
os.environ["JAVA_HOME"] = str(Path(sys.prefix) / "Library" / "jre")

# Repo paths + frame:
data_path = Path.cwd().resolve().parent / "data"
download_path = Path.cwd().resolve().parent / "download"

In [ ]:
# TLE paths for LAGEOS-1:
tle_path = download_path / "tle" / "8820.json"
df = pd.read_json(tle_path)

df.columns

In [ ]:
# Sort TLEs + remove duplicate epochs:
df["EPOCH"] = pd.to_datetime(df["EPOCH"], utc=True).dt.tz_localize(None)
df["CREATION_DATE"] = pd.to_datetime(df["CREATION_DATE"], utc=True)
df = df[(df["EPOCH"] >= "2025-01-01") & (df["EPOCH"] < "2026-01-01")]  # only 2025
df = df.sort_values(["EPOCH", "CREATION_DATE"])
df = df.drop_duplicates("EPOCH", keep="last")  # drop stalest TLEs at duplicate epochs

# Extract mean elements from the TLEs:
e = df["ECCENTRICITY"].to_numpy(dtype=float)
i = df["INCLINATION"].to_numpy(dtype=float)
raan = df["RA_OF_ASC_NODE"].to_numpy(dtype=float)
omega = df["ARG_OF_PERICENTER"].to_numpy(dtype=float)
M = np.deg2rad(df["MEAN_ANOMALY"].to_numpy(dtype=float))

df.head(5)

## 1. Mean Elements:

The idea for the mean elements is to extract the raw TLE elements, add the secular, long-period and short-period periodic updates, and then plot the corrected elements over time. This is akin to almost manually performing SGP4 via the theory provided in Hoots et al. (2004). These will be then compared to the classical osculating elements i.e., the instantaneous Keplerian orbital elements yielded by SGP4 from the calculated ($\vec{r}, \vec{v}$) quantities.

The TLE elements at each epoch will be obtained and then updated at regular 1-hour time intervals, until the next TLE that is available.
 
### Kozai-to-Brouwer Convention Recovery at TLE Epoch

Having extracted the TLE $e, i, \Omega, \omega$, and $M$ elements, in accordance with Hoots et al. (2004) specifically Appendix B (Initialisation), we will initialise the Brouwer semi-major axis $a_0''$.

In [ ]:
# Recover SGP4 mean semimajor axis from Hoots et al. (2004):
n0 = df["MEAN_MOTION"].to_numpy(dtype=float)*2*np.pi/1440
ke = 0.0743669161  # Earth radii^(3/2)/min
J2 = 1.082616e-3
Re = 6378.135 # Earth radius [km]

a1 = (ke/n0)**(2/3)
beta = np.sqrt(1 - e**2)
delta1 = 0.75*J2*(3*np.cos(np.deg2rad(i))**2 - 1)/(a1**2 * beta**3)
a2 = a1*(1 - delta1/3 - delta1**2 - 134*delta1**3/81)
delta0 = 0.75*J2*(3*np.cos(np.deg2rad(i))**2 - 1) / (a2**2*beta**3)
n_mean = n0 / (1 + delta0)
a = (ke / n_mean)**(2/3)*Re  # recovered semi-major axis

### Secular Updates for Earth Zonal Gravity

### Secular Updates for Lunisolar Gravity Effects ($e, \Omega, \omega, M$)

### Secular Updates for Resonance Effects of Earth Gravity ($M$)

### Secular Updates for Remaining Atmospheric Drag ($a, e$)

### Long-Period Periodic Update for Lunisolar Gravity Effects ($e, i, \Omega, \omega, M$)

### Short-Period Periodic Update for Earth's Gravity Effects ($e$)

## 2. Osculating Elements:

### Deriving from $(\vec{r}_{\text{TLE}}, \vec{v}_{\text{TLE}})$ 

Loading the pre-derived positional and velocity vectors $\vec{r}_{\text{TLE}}$ and $\vec{v}_{\text{TLE}}$ calculated and exported into CSV files in a [previous noteboook](01_tle_sp3_residuals.ipynb), we can now plug these into equations for deriving osculating orbital elements, where:

$$
\vec{r}_{\mathrm{TLE}} =
\begin{bmatrix}x\\y\\z\end{bmatrix},
\qquad
\vec{v}_{\mathrm{TLE}} =
\begin{bmatrix}v_x\\v_y\\v_z\end{bmatrix}.
$$


The classical osculating orbital elements are recovered from these vectors using the two-body relations, where all angular elements are in degrees. Using the vis-viva equation, the osculating semi-major axis can be obtained via

$$
a_{\mathrm{osc}} =
\frac{1}{
\frac{2}{r}-\frac{v^2}{\mu}
} \; \; \; \; \; (1)
$$

The eccentricity vector is

$$
\vec{e} =
\frac{\vec{v}_{\mathrm{TLE}}\times\vec{h}}{\mu}
-\frac{\vec{r}_{\mathrm{TLE}}}{r}
$$

Thus, taking the magnitude yields the osculating eccentricity such that

$$
e_{\mathrm{osc}}=\|\vec{e}\| \; \; \; \; \; (2)
$$

The osculating inclination can be found via

$$
i_{\mathrm{osc}} =
\cos^{-1}\left(\frac{h_z}{h}\right) \; \; \; \; \; (3)
$$

The right ascension of the ascending node (RAAN) is obtained through

$$
\Omega_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(\frac{N_x}{N}\right), & N_y\geq0,\; \; \; \; \; (4)\\
2\pi-\cos^{-1}\left(\frac{N_x}{N}\right), & N_y<0.
\end{cases}
$$

The argument of periapsis can be calculated such that
$$
\omega_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(\frac{\mathbf N\cdot\mathbf e}{Ne}\right), & e_z\geq0, \; \; \; \; \; (5)\\
2\pi-\cos^{-1}\left(\frac{\mathbf N\cdot\mathbf e}{Ne}\right),
& e_z<0.
\end{cases}
$$

Finally, the true anomaly of a satellite is obtained using
$$
\theta_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(
\frac{\mathbf e\cdot\mathbf r_{\mathrm{TLE}}}{er}
\right), & \mathbf r_{\mathrm{TLE}}\cdot\mathbf v_{\mathrm{TLE}}\geq0, \; \; \; \; \; (6)\\
2\pi-\cos^{-1}\left(
\frac{\mathbf e\cdot\mathbf r_{\mathrm{TLE}}}{er}
\right), & \mathbf r_{\mathrm{TLE}}\cdot\mathbf v_{\mathrm{TLE}}<0.
\end{cases}
$$

In [ ]:
# Load precalculated SGP4 LAGEOS-1 states:
tle_states = pd.read_csv(data_path/"states"/"lageos1_tle_2025.csv", parse_dates=["epoch_utc"])
print("Number of propagated states:", len(tle_states))
tle_states.head()

In [ ]:
# Get instataneous r and v + useful vectors:
r = np.sqrt(tle_states['x_m']**2 + tle_states['y_m']**2 + tle_states['z_m']**2).to_numpy()
v = np.sqrt(tle_states['vx_m_s']**2 + tle_states['vy_m_s']**2 + tle_states['vz_m_s']**2).to_numpy()
r_vec = tle_states[['x_m', 'y_m', 'z_m']].to_numpy()
v_vec = tle_states[['vx_m_s', 'vy_m_s', 'vz_m_s']].to_numpy()

mu = (6.6743*10**-11)*(5.972*10**24)  # mu = GM
h = np.cross(r_vec, v_vec)  # momentum vector
h_mag = np.linalg.norm(h, axis=1)
z_hat = np.array([0, 0, 1])
N = np.cross(z_hat, h)  # nodal vector (direction of AN)
N_mag = np.linalg.norm(N, axis=1)

# Semi-major axis:
a = 1/((2/r)-(v**2/mu))  # using (1)

# Eccentricity:
e_vec = np.cross(v_vec, h)/mu - r_vec/r[:, None]
e = np.linalg.norm(e_vec, axis=1)  # using (2)

# Inclination:
i = np.degrees(np.arccos(np.clip(h[:, 2]/h_mag, -1, 1)))  # using (3)

# RAAN:
raan = np.degrees(np.arctan2(N[:, 1], N[:, 0]))%360  # using (4)

# Argument of periapsis:
omega = np.degrees(np.arccos(np.clip(np.sum(N*e_vec, axis=1) / (N_mag*e), -1, 1)))
omega = np.where(e_vec[:, 2] < 0, 360 - omega, omega)  # using (5)

# True anomaly:
theta = np.degrees(np.arccos(np.clip(np.sum(e_vec*r_vec, axis=1)/(e*r), -1, 1)))
theta = np.where(np.sum(r_vec*v_vec, axis=1) < 0, 360 - theta, theta)  # using (6)

# Saving osculating orbital elements to CSV:
osc_elements = pd.DataFrame({
    "epoch_utc": tle_states["epoch_utc"],
    "a_osc_km": a*0.001,
    "e_osc": e,
    "i_osc_deg": i,
    "raan_osc_deg": raan,
    "omega_osc_deg": omega,
    "theta_osc_deg": theta
})

osc_path = data_path / "osc-elements"
osc_path.mkdir(parents=True, exist_ok=True)
osc_elements.to_csv(osc_path / "lageos1_osc_2025.csv", index=False)

print("Number of osculating element sets:", len(osc_elements))


## 3. Comparison:

### Plotting Mean and Osculating Orbit Elements over Time

In [ ]:
# Running script from Jupyter notebook + bypassing kernel crash:
conda_path = Path(sys.prefix)

# Include Conda's DLL directories
env = os.environ.copy()
env["PATH"] = os.pathsep.join([
    str(conda_path),
    str(conda_path / "Library" / "bin"),
    str(conda_path / "Library" / "usr" / "bin"),
    str(conda_path / "Scripts"),
    env.get("PATH", "")
])

# Run plotting script in a separate process
subprocess.run([sys.executable, "plot_osc_mean_elements.py"], env=env, check=True)

In [ ]:
from IPython.display import Image, display
display(Image(filename="figures/a_lageos1.png", width=800))

In [ ]:
display(Image(filename="figures/e_lageos1.png", width=800))

In [ ]:
display(Image(filename="figures/i_lageos1.png", width=800))

In [ ]:
display(Image(filename="figures/omega_lageos1.png", width=800))

In [ ]:
display(Image(filename="figures/raan_lageos1.png", width=800))

In [ ]:
display(Image(filename="figures/theta_lageos1.png", width=800))

### Bibliography

Hoots, F. R., Schumacher Jr., P. W., Glover, R. A. (2004). _History of Analytical Orbit Modeling in the U.S. Space Surveillance System_. Journal of Guidance, Control, and Dynamics, Vol. 27, No. 2.